In [1]:
# Import PySpark
from pyspark.sql import SparkSession

#Create SparkSession
spark = SparkSession.builder.appName('SparkByExamples.com').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/04/15 02:21:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import os
import json
import gzip
import pandas as pd
import datetime
import hashlib
from urllib.request import urlopen
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, IndexToString
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder

In [3]:
# !wget http://deepyeti.ucsd.edu/jianmo/amazon/sample/meta_Computers.json.gz
!wget --no-check-certificate https://jmcauley.ucsd.edu/data/amazon_v2/categoryFiles/Magazine_Subscriptions.json.gz

--2023-04-15 02:22:02--  https://jmcauley.ucsd.edu/data/amazon_v2/categoryFiles/Magazine_Subscriptions.json.gz
Resolving jmcauley.ucsd.edu (jmcauley.ucsd.edu)... 137.110.160.73
Connecting to jmcauley.ucsd.edu (jmcauley.ucsd.edu)|137.110.160.73|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request sent, awaiting response... 200 OK
Length: 12875273 (12M) [application/x-gzip]
Saving to: 'Magazine_Subscriptions.json.gz.2'

Magazine_Subscripti 100%[===================>]  12.28M   339KB/s    in 14s     

2023-04-15 02:22:16 (930 KB/s) - 'Magazine_Subscriptions.json.gz.2' saved [12875273/12875273]



In [4]:
### load the meta data

data = []
with gzip.open('Magazine_Subscriptions.json.gz') as f:
    for l in f:
        data.append(json.loads(l.strip()))
    
# total length of list, this number equals total number of products
print(len(data))

# first row of the list
print(data[0])

89689
{'overall': 5.0, 'vote': '9', 'verified': False, 'reviewTime': '11 8, 2001', 'reviewerID': 'AH2IFH762VY5U', 'asin': 'B00005N7P0', 'reviewerName': 'ted sedlmayr', 'reviewText': 'for computer enthusiast, MaxPC is a welcome sight in your mailbox. i can remember for years savorying every page of "boot" (as it was called in beginning) as i was (and still am) obcessed with PC\'s. Anyone, from advanced users - to beginners looking for knowledge - can profit from every issue of MaxPC. the icing on the cake is the subscription that comes with a CD-ROM as it is packed with demos, utilities, and other useful apps (very helpful for those not blessed with broadband connections). Until I discovered the community of hardware enthusiast web sites, MaxPC, formerly "boot", was my only really informative source for computing news and articles. To this day, i consider my subscription to it worth more than 10 subscriptions to most other computing mags. I can\'t wait until they merge with DVD media an

In [5]:
# convert list into pandas dataframe

df = pd.DataFrame.from_dict(data)

df['reviewTime'] = pd.to_datetime(df['reviewTime'], format='%m %d, %Y')
df['vote'] = df['vote'].str.replace(',', '').fillna(0).astype(int)
df["style"] = df["style"].apply(lambda x: {} if pd.isna(x) else x)
df = df.fillna('')

df = df[:1000] ###test with small subset!
df

,overall,vote,verified,reviewTime,reviewerID,asin,reviewerName,reviewText,summary,unixReviewTime,style,image
0,5.0,9,False,2001-11-08,AH2IFH762VY5U,B00005N7P0,ted sedlmayr,"for computer enthusiast, MaxPC is a welcome si...","AVID READER SINCE ""boot"" WAS THE NAME",1005177600,{},
1,5.0,9,False,2001-10-31,AOSFI0JEYU4XM,B00005N7P0,Amazon Customer,Thank god this is not a Ziff Davis publication...,The straight scoop,1004486400,{},
2,3.0,14,False,2007-03-24,A3JPFWKS83R49V,B00005N7OJ,Bryan Carey,Antiques Magazine is a publication made for an...,"Antiques Magazine is Good, but not for Everyone",1174694400,{'Format:': ' Print Magazine'},
3,5.0,13,False,2006-11-10,A19FKU6JZQ2ECJ,B00005N7OJ,Patricia L. Porada,This beautiful magazine is in itself a work of...,THE DISCERNING READER,1163116800,{'Format:': ' Print Magazine'},
4,5.0,0,True,2014-07-14,A25MDGOMZ2GALN,B00005N7P0,Alvey,A great read every issue.,Five Stars,1405296000,{},
...,...,...,...,...,...,...,...,...,...,...,...,...
995,2.0,0,True,2017-07-05,ABWKZTT8PMPIR,B00005N7RD,Slick,Really only bought this because of the cheap p...,Good for beginners,1499212800,{'Format:': ' Print Magazine'},
996,2.0,0,False,2017-06-24,AQ3T1IVQMOMIV,B00005N7RD,exDTguy,I have been a Men's Health subscriber for many...,Formerly An Excellent Publication,1498262400,{'Format:': ' Print Magazine'},
997,2.0,3,True,2017-05-22,AENTGS8TB0JAD,B00005N7RD,jesseasg,I'm sorry I subscribed to this magazine. I'm v...,"Not what I was hoping for,disappointing",1495411200,{'Format:': ' Print Magazine'},
998,5.0,0,True,2017-05-17,A1OQ2MFY97V991,B00005N7RD,Hank of Palm Springs,"a great magazine even for an ""old timer""",Men's Health magazine,1494979200,{'Format:': ' Print Magazine'},


In [6]:
from pyspark.sql.types import *

# Define the schema for the RDD
schema = StructType([
    StructField("overall", DoubleType(), True),
    StructField("vote", IntegerType(), True),
    StructField("verified", BooleanType(), True),
    StructField("reviewTime", TimestampType(), True),
    StructField("reviewerID", StringType(), True),
    StructField("asin", StringType(), True),
    StructField("reviewerName", StringType(), True),
    StructField("reviewText", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("unixReviewTime", StringType(), True),
    StructField("style", MapType(StringType(), StringType(), True), True),
    StructField("image", StringType(), True),
])

reviews = spark.createDataFrame(df, schema=schema) 

In [7]:
# convert reviewerID and asin to numeric type
stringIndexer = StringIndexer(inputCols=['reviewerID','asin'], outputCols=['reviewer','product'])
model = stringIndexer.fit(reviews)
reviews = model.transform(reviews)

Error processing line 3 of /Users/yuhanwang/opt/anaconda3/lib/python3.9/site-packages/googleapis_common_protos-1.58.0-py3.9-nspkg.pth:

  Traceback (most recent call last):
    File "/Users/yuhanwang/opt/anaconda3/lib/python3.9/site.py", line 169, in addpackage
      exec(line)
    File "<string>", line 1, in <module>
    File "<frozen importlib._bootstrap>", line 562, in module_from_spec
  AttributeError: 'NoneType' object has no attribute 'loader'

Remainder of file ignored


In [8]:
reviews.printSchema()
reviews.show(5)

root
 |-- overall: double (nullable = true)
 |-- vote: integer (nullable = true)
 |-- verified: boolean (nullable = true)
 |-- reviewTime: timestamp (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- reviewerName: string (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- unixReviewTime: string (nullable = true)
 |-- style: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- image: string (nullable = true)
 |-- reviewer: double (nullable = false)
 |-- product: double (nullable = false)

+-------+----+--------+-------------------+--------------+----------+------------------+--------------------+--------------------+--------------+--------------------+-----+--------+-------+
|overall|vote|verified|         reviewTime|    reviewerID|      asin|      reviewerName|          reviewText|             summary|unixReviewTime|               style|im

## Running ALS on Reviews

In [9]:
# split the data to training and test parts (80%-20% ratio)
train, test = reviews.randomSplit([0.8, 0.2], seed=10)

print ("N train", train.cache().count())
print ("N test", test.cache().count())

N train 794
N test 206


In [10]:
als = ALS(
    rank=10,
    maxIter=15,
    regParam=0.05,
    userCol='reviewer',
    itemCol='product',
    ratingCol='overall',  
    coldStartStrategy='drop',
    nonnegative=True
)

In [11]:
# fit the model
model = als.fit(train)

23/04/15 02:22:34 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
23/04/15 02:22:34 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS


In [12]:
# performing predictions using the testing data
prediction = model.transform(test)
prediction.cache().show()

+-------+----+--------+-------------------+--------------+----------+------------------+--------------------+--------------------+--------------+--------------------+-----+--------+-------+----------+
|overall|vote|verified|         reviewTime|    reviewerID|      asin|      reviewerName|          reviewText|             summary|unixReviewTime|               style|image|reviewer|product|prediction|
+-------+----+--------+-------------------+--------------+----------+------------------+--------------------+--------------------+--------------+--------------------+-----+--------+-------+----------+
|    1.0|   5|   false|2004-03-09 00:00:00| A5RHZE7B8SV5Q|B00005N7P0|    gorillazfan249|Maximum PC's slog...|More like half an...|    1078790400|                  {}|     |     6.0|    1.0|  1.103332|
|    4.0|   2|   false|2007-09-24 00:00:00|A3JPFWKS83R49V|B00005N7P0|       Bryan Carey|Maximum PC is a m...|Maximizing Electr...|    1190592000|                  {}|     |     0.0|    1.0|0.80370

In [13]:
# evaluate the model by computing the RMSE on the test data
evaluator_als = RegressionEvaluator(metricName="rmse", 
                                labelCol="overall", 
                                predictionCol="prediction")

rmse = evaluator_als.evaluate(prediction)
print("Root-mean-square error: " + str(rmse))

Root-mean-square error: 2.110396352925673


In [20]:
# tune model using grid search
param_grid = ParamGridBuilder().addGrid(als.rank, [13,14,15]) \
                               .addGrid(als.maxIter, [17,18,19]) \
                               .addGrid(als.regParam, [.01,.15,.1]).build()

# cross-validation setting (this replaces the baseline model during training)
tvs = TrainValidationSplit(estimator=als,
                           estimatorParamMaps=param_grid,
                           evaluator=evaluator_als)

In [21]:
model = tvs.fit(train)

best_model = model.bestModel
prediction = best_model.transform(test)

In [22]:
# checking model accuracy
rmse = evaluator_als.evaluate(prediction)
# printing evaluation metrics and model parameters
print("Root-mean-square error: " + str(rmse))
print("**Best model**")
print("  Rank:", best_model.rank)
print("  MaxIter:", best_model._java_obj.parent().getMaxIter())
print("  RegParam:", best_model._java_obj.parent().getRegParam())

Root-mean-square error: 1.718159127223401
**Best model**
  Rank: 14
  MaxIter: 17
  RegParam: 0.1
